# The design matrix

The scalar formulas are the two-by-two case of a matrix equation. Writing that equation now means multiple regression will not need a new theory, only a wider matrix.


## The model in matrix form

Stack the responses into a vector $y$ and build the design matrix $X$ with a column of ones and a column of the predictor. For the three-point table,

$$
y = \begin{bmatrix} 2 \\ 3 \\ 5 \end{bmatrix}, \qquad
X = \begin{bmatrix} 1 & 1 \\ 1 & 2 \\ 1 & 3 \end{bmatrix}, \qquad
\beta = \begin{bmatrix} b_0 \\ b_1 \end{bmatrix}.
$$

The predictions are the matrix product $X\beta$, and

$$
S(\beta) = (y - X\beta)^{\mathsf T}(y - X\beta) = \|y - X\beta\|^2.
$$

The gradient with respect to $\beta$ is $-2X^{\mathsf T}(y - X\beta)$. Setting it equal to zero gives the normal equations

$$
X^{\mathsf T}X\,\beta = X^{\mathsf T}y.
$$

When $X^{\mathsf T}X$ is invertible, the unique solution is $\widehat\beta = (X^{\mathsf T}X)^{-1}X^{\mathsf T}y$.


## Multiplying the three-point matrices

$$
X^{\mathsf T}X
= \begin{bmatrix} 1 & 1 & 1 \\ 1 & 2 & 3 \end{bmatrix}
\begin{bmatrix} 1 & 1 \\ 1 & 2 \\ 1 & 3 \end{bmatrix}
= \begin{bmatrix} 3 & 6 \\ 6 & 14 \end{bmatrix}, \qquad
X^{\mathsf T}y
= \begin{bmatrix} 10 \\ 23 \end{bmatrix}.
$$

The system $X^{\mathsf T}X\,\beta = X^{\mathsf T}y$ is exactly the pair of scalar normal equations:

$$
\begin{aligned}
3b_0 + 6b_1 &= 10, \\
6b_0 + 14b_1 &= 23.
\end{aligned}
$$

Divide the first equation by $3$, then eliminate. From $b_0 + 2b_1 = 10/3$,

$$
6b_0 + 12b_1 = 20.
$$

Subtract that from the second equation:

$$
2b_1 = 3 \Longrightarrow b_1 = \dfrac{3}{2}, \qquad
b_0 = \dfrac{10}{3} - 3 = \dfrac{1}{3}.
$$

The determinant confirms that the inverse exists:

$$
\det(X^{\mathsf T}X) = 3\cdot 14 - 6\cdot 6 = 6,
$$

and

$$
(X^{\mathsf T}X)^{-1} = \dfrac{1}{6}\begin{bmatrix} 14 & -6 \\ -6 & 3 \end{bmatrix}
= \begin{bmatrix} 7/3 & -1 \\ -1 & 1/2 \end{bmatrix}.
$$

One matrix product recovers both coefficients:

$$
(X^{\mathsf T}X)^{-1}X^{\mathsf T}y
= \begin{bmatrix} 7/3 & -1 \\ -1 & 1/2 \end{bmatrix}
\begin{bmatrix} 10 \\ 23 \end{bmatrix}
= \begin{bmatrix} 70/3 - 23 \\ -10 + 23/2 \end{bmatrix}
= \begin{bmatrix} 1/3 \\ 3/2 \end{bmatrix}.
$$


In [1]:
# Build X, form the normal equations, and solve them by the inverse.
from fractions import Fraction
import numpy as np
X = np.array([
    [Fraction(1), Fraction(1)],
    [Fraction(1), Fraction(2)],
    [Fraction(1), Fraction(3)],
])
y = np.array([Fraction(2), Fraction(3), Fraction(5)])
gram = X.T @ X
right = X.T @ y
inverse = np.array([
    [Fraction(7, 3), Fraction(-1)],
    [Fraction(-1), Fraction(1, 2)],
])
beta = inverse @ right
print("X'X")
print(gram)
print("X'y", right)
print("beta", beta)
assert np.all(gram == np.array([[3, 6], [6, 14]], dtype=object))
assert list(right) == [Fraction(10), Fraction(23)]
assert np.all(inverse @ gram == np.eye(2, dtype=object))
assert list(beta) == [Fraction(1, 3), Fraction(3, 2)]


X'X
[[Fraction(3, 1) Fraction(6, 1)]
 [Fraction(6, 1) Fraction(14, 1)]]
X'y [Fraction(10, 1) Fraction(23, 1)]
beta [Fraction(1, 3) Fraction(3, 2)]


## One change: swap in the five-point design

The same multiplication on $x = 1, 2, 3, 4, 5$ and $y = 4, 4, 7, 8, 12$ produces

$$
X^{\mathsf T}X = \begin{bmatrix} 5 & 15 \\ 15 & 55 \end{bmatrix}, \qquad
X^{\mathsf T}y = \begin{bmatrix} 35 \\ 125 \end{bmatrix}.
$$

Those entries are the sums $n$, $\sum x$, $\sum x^2$, $\sum y$, and $\sum xy$. Solving gives $\beta = (1, 2)^{\mathsf T}$, the line already fitted from the sums. The matrix statement did not create a new estimator.


In [2]:
# The five-point normal matrix is the table of sums, and its solution is (1, 2).
import numpy as np
x = np.array([1.0, 2, 3, 4, 5])
y = np.array([4.0, 4, 7, 8, 12])
X = np.column_stack([np.ones(5), x])
gram = X.T @ X
right = X.T @ y
beta = np.linalg.solve(gram, right)
print("X'X")
print(gram)
print("X'y", right)
print("beta", beta)
assert np.array_equal(gram, np.array([[5.0, 15], [15, 55]]))
assert np.allclose(right, [35, 125])
assert np.allclose(beta, [1, 2])


X'X
[[ 5. 15.]
 [15. 55.]]
X'y [ 35. 125.]
beta [1. 2.]


## Pitfall

The product $X\beta$ is not the same object as $\beta^{\mathsf T}X$, and a row vector of predictors cannot be multiplied on the left by a column $\beta$. For the first observation the prediction is the row $[1,\ 1]$ times the column $(1/3,\ 3/2)^{\mathsf T}$:

$$
\begin{bmatrix} 1 & 1 \end{bmatrix}
\begin{bmatrix} 1/3 \\ 3/2 \end{bmatrix}
= \dfrac{1}{3} + \dfrac{3}{2} = \dfrac{11}{6}.
$$

NumPy will reject a product whose inner dimensions disagree rather than quietly align them the way a spreadsheet might.


In [3]:
# A row of X times beta is a prediction. The swapped product is not defined.
import numpy as np
beta = np.array([1 / 3, 3 / 2])
row = np.array([1.0, 1.0])
print("prediction", row @ beta)
assert abs(row @ beta - 11 / 6) < 1e-12
try:
    print(beta @ np.array([row, row, row]))
    raise AssertionError("the swapped product should fail")
except ValueError as exc:
    print("shapes refused:", exc.__class__.__name__)


prediction 1.8333333333333333
shapes refused: ValueError


## Summary

- Simple regression is $y \approx X\beta$ with a column of ones beside the column $x$.
- The least-squares coefficient is the solution of $X^{\mathsf T}X\,\beta = X^{\mathsf T}y$.
- On the three-point table that solution is $(1/3,\ 3/2)$. On the five-point table it is $(1,\ 2)$.
